# 01. Start here: a safe pySTAMPS pipeline walkthrough

This notebook is a small operational walkthrough. It creates a throwaway Stage-1 dataset, checks status, previews work with a dry-run, executes Stage 1, and checks the outputs.

Nothing here assumes a completed `inputs_and_outputs` dataset. The demo writes only to a temporary scratch directory so you can rerun it safely.

In [1]:
from pathlib import Path
import json
import tempfile

import numpy as np

from pystamps.config import RunConfig
from pystamps.input_contracts import describe_stage_inputs
from pystamps.io.mat import read_mat
from pystamps.pipeline.stages import run_pipeline
from pystamps.pipeline.types import PipelineContext
from pystamps.status import collect_status

REPO_ROOT = Path.cwd()
SCRATCH_ROOT = Path(tempfile.mkdtemp(prefix="pystamps_stage1_demo_"))
DATASET = SCRATCH_ROOT / "stage1_demo_dataset"
PATCH = DATASET / "PATCH_1"

print(f"Repository root: {REPO_ROOT}")
print(f"Scratch dataset: {DATASET}")
print("Why this matters: pipeline stages write artifacts into the dataset tree, so tutorials should run on scratch data or a copy.")

Repository root: /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS
Scratch dataset: /tmp/pystamps_stage1_demo_s5vel2f8/stage1_demo_dataset
Why this matters: pipeline stages write artifacts into the dataset tree, so tutorials should run on scratch data or a copy.


## 1. Build the smallest useful Stage-1 dataset

Stage 1 converts raw patch candidate files into `ps1.mat`, `ph1.mat`, `bp1.mat`, and related patch artifacts. The inputs below are synthetic, but they follow the same file contract that a SNAP-to-StaMPS export would provide.

In [2]:
PATCH.mkdir(parents=True)
(DATASET / "patch.list").write_text("PATCH_1\n", encoding="utf-8")
(DATASET / "width.txt").write_text("20\n", encoding="utf-8")
(DATASET / "len.txt").write_text("10\n", encoding="utf-8")

# Four candidate points. Columns are candidate id, row, column.
ij = np.array([
    [1, 2, 3],
    [2, 4, 8],
    [3, 6, 13],
    [4, 8, 18],
], dtype=float)
np.savetxt(PATCH / "pscands.1.ij", ij, fmt="%.0f")

# Lon/lat pairs are binary float32 in the Stage-1 contract.
lonlat = np.array([
    [11.000, 44.000],
    [11.005, 44.003],
    [11.010, 44.006],
    [11.015, 44.009],
], dtype=">f4")
lonlat.tofile(PATCH / "pscands.1.ll")

# Three slave interferograms. Stage 1 inserts the master column automatically.
phase = np.array([
    [1.0 + 0.0j, 0.9 + 0.1j, 0.8 + 0.2j],
    [0.8 + 0.2j, 0.7 + 0.3j, 0.6 + 0.4j],
    [0.6 + 0.1j, 0.5 + 0.2j, 0.4 + 0.3j],
    [0.4 + 0.3j, 0.3 + 0.4j, 0.2 + 0.5j],
], dtype=np.complex64)
phase_file_blocks = []
for col in range(phase.shape[1]):
    interleaved = np.empty(phase.shape[0] * 2, dtype="<f4")
    interleaved[0::2] = phase[:, col].real
    interleaved[1::2] = phase[:, col].imag
    phase_file_blocks.append(interleaved)
np.concatenate(phase_file_blocks).tofile(PATCH / "pscands.1.ph")

# Optional amplitude-dispersion values become da1.mat.
np.savetxt(PATCH / "pscands.1.da", np.array([0.20, 0.25, 0.30, 0.35]), fmt="%.3f")

# Three slave dates plus one master date. bperp has one value per slave interferogram.
np.savetxt(PATCH / "day.1.in", np.array([20200112, 20200124, 20200205]), fmt="%.0f")
np.savetxt(PATCH / "master_day.1.in", np.array([20200101]), fmt="%.0f")
np.savetxt(PATCH / "bperp.1.in", np.array([-120.0, 35.0, 90.0]), fmt="%.3f")

print("Created one patch with 4 candidates and 3 slave interferograms.")
print("Important files now present:")
for path in sorted(PATCH.iterdir()):
    print(f"- {path.relative_to(DATASET)}")

Created one patch with 4 candidates and 3 slave interferograms.
Important files now present:
- PATCH_1/bperp.1.in
- PATCH_1/day.1.in
- PATCH_1/master_day.1.in
- PATCH_1/pscands.1.da
- PATCH_1/pscands.1.ij
- PATCH_1/pscands.1.ll
- PATCH_1/pscands.1.ph


## 2. Check status before running

`status` answers the first operational question: what has already been produced? A new raw Stage-1 dataset should show patch stage `0` and merged stage `0`.

In [3]:
def print_status(label, dataset):
    status = collect_status(dataset)
    print(label)
    print(f"dataset: {status.dataset}")
    print(f"merged stage: {status.merged_stage}")
    for patch in status.patch_statuses:
        print(f"{patch.patch}: patch stage {patch.stage}")

print("CLI equivalent: uv run pystamps status --dataset <scratch dataset>")
print_status("Current status:", DATASET)
print("Why this matters: status is artifact-based, so it tells you whether a stage can be skipped or resumed.")

CLI equivalent: uv run pystamps status --dataset <scratch dataset>
Current status:
dataset: /tmp/pystamps_stage1_demo_s5vel2f8/stage1_demo_dataset
merged stage: 0
PATCH_1: patch stage 0
Why this matters: status is artifact-based, so it tells you whether a stage can be skipped or resumed.


## 3. Dry-run Stage 1

A dry-run previews the work without writing outputs. Use this before running on a real copied dataset.

In [4]:
def run_stage(start, end, dry_run):
    context = PipelineContext(
        dataset_root=DATASET,
        run_config=RunConfig(),
        start_step=start,
        end_step=end,
        dry_run=dry_run,
    )
    return run_pipeline(context)

def print_report(report):
    for result in report.results:
        print(
            f"stage {result.stage_id} | {result.scope} | {result.target} | "
            f"{result.status}: {result.details}"
        )
    print(f"failures: {len(report.failures)}")

dry_report = run_stage(1, 1, dry_run=True)
print("CLI equivalent: uv run pystamps run --dataset <scratch dataset> --start-step 1 --end-step 1 --dry-run")
print_report(dry_report)
print("Why this matters: planned means pySTAMPS found missing Stage-1 outputs and knows what it would create.")

CLI equivalent: uv run pystamps run --dataset <scratch dataset> --start-step 1 --end-step 1 --dry-run
stage 1 | patch | PATCH_1 | planned: Would produce ps1.mat
failures: 0
Why this matters: planned means pySTAMPS found missing Stage-1 outputs and knows what it would create.


## 4. Execute Stage 1

Now run exactly the stage that was planned. This writes Stage-1 patch artifacts into the scratch patch directory.

In [5]:
run_report = run_stage(1, 1, dry_run=False)
print("CLI equivalent: uv run pystamps run --dataset <scratch dataset> --start-step 1 --end-step 1")
print_report(run_report)
print("Why this matters: completed means Stage 1 consumed raw candidate files and wrote the first pySTAMPS artifacts.")

CLI equivalent: uv run pystamps run --dataset <scratch dataset> --start-step 1 --end-step 1
stage 1 | patch | PATCH_1 | completed: Stage 1 created ps1/ph1 for 4 candidates
failures: 0
Why this matters: completed means Stage 1 consumed raw candidate files and wrote the first pySTAMPS artifacts.


## 5. Inspect what changed

After execution, status should report patch stage `1`. The most useful immediate check is that `ps1.mat` and `ph1.mat` exist and have shapes matching the synthetic inputs.

In [6]:
print_status("Status after Stage 1:", DATASET)

created = ["ps1.mat", "ph1.mat", "bp1.mat", "da1.mat", "psver.mat"]
print("\nStage-1 artifacts:")
for name in created:
    path = PATCH / name
    print(f"- {name}: {'present' if path.exists() else 'missing'}")

ps1 = read_mat(PATCH / "ps1.mat")
ph1 = read_mat(PATCH / "ph1.mat")
bp1 = read_mat(PATCH / "bp1.mat")

print("\nKey payload checks:")
print(f"ps1.n_ps: {int(np.asarray(ps1['n_ps']).reshape(-1)[0])} candidates")
print(f"ps1.n_ifg: {int(np.asarray(ps1['n_ifg']).reshape(-1)[0])} images including inserted master")
print(f"ph1.ph shape: {np.asarray(ph1['ph']).shape}")
print(f"bp1.bperp_mat shape: {np.asarray(bp1['bperp_mat']).shape}")
print("Why this matters: Stage 1 has normalized raw patch inputs into MATLAB-compatible artifacts for later stages.")

Status after Stage 1:
dataset: /tmp/pystamps_stage1_demo_s5vel2f8/stage1_demo_dataset
merged stage: 0
PATCH_1: patch stage 1

Stage-1 artifacts:
- ps1.mat: present
- ph1.mat: present
- bp1.mat: present
- da1.mat: present
- psver.mat: present

Key payload checks:
ps1.n_ps: 4 candidates
ps1.n_ifg: 4 images including inserted master
ph1.ph shape: (4, 4)
bp1.bperp_mat shape: (4, 3)
Why this matters: Stage 1 has normalized raw patch inputs into MATLAB-compatible artifacts for later stages.


## 6. Dry-run again to see resumability

The same dry-run now reports `skipped_existing` because `ps1.mat` already exists. This is how pySTAMPS avoids redoing completed stages.

In [7]:
rerun_dry_report = run_stage(1, 1, dry_run=True)
print_report(rerun_dry_report)
print("Why this matters: a copied real dataset can be resumed from the artifacts already present in each patch.")

stage 1 | patch | PATCH_1 | skipped_existing: ps1.mat present
failures: 0
Why this matters: a copied real dataset can be resumed from the artifacts already present in each patch.


## 7. Stage map for next steps

This notebook stops at Stage 1 because the tiny synthetic data is meant to teach the workflow, not produce scientific results. For real processing, run on a copy of a valid StaMPS dataset and expand the stage range only after status and dry-run look right.

In [8]:
print("Stages and scopes:")
for row in describe_stage_inputs("all"):
    stage = row["stage"]
    title = row["title"]
    scope = "patch" if stage <= 5 else "merged"
    print(f"- Stage {stage} ({scope}): {title}")

print("\nNext operational pattern:")
print("1. Copy your dataset.")
print("2. Run status on the copy.")
print("3. Dry-run the intended stage range.")
print("4. Execute the smallest stage range you need.")
print("5. Run status again and inspect the new artifacts.")
print(f"\nScratch dataset for this executed notebook: {DATASET}")

Stages and scopes:
- Stage 1 (patch): Load and organize raw candidate inputs
- Stage 2 (patch): Estimate phase model and coherence per patch
- Stage 3 (patch): Select persistent scatterers
- Stage 4 (patch): Weed noisy or redundant candidates
- Stage 5 (patch): Merge patch outputs into one dataset view
- Stage 6 (merged): Unwrap the merged phase products
- Stage 7 (merged): Estimate slow trends and correction terms
- Stage 8 (merged): Apply final space-time filtering

Next operational pattern:
1. Copy your dataset.
2. Run status on the copy.
3. Dry-run the intended stage range.
4. Execute the smallest stage range you need.
5. Run status again and inspect the new artifacts.

Scratch dataset for this executed notebook: /tmp/pystamps_stage1_demo_s5vel2f8/stage1_demo_dataset
